# Adquisición, cruce y calidad de datos — Dengue y lluvias 2021-2025

**Pregunta del proyecto:** ¿Qué municipios y semanas epidemiológicas concentran más notificaciones de dengue entre 2021 y 2025, y qué aumentos recurrentes conviene contrastar con las temporadas de lluvia para que las autoridades de salud prioricen la vigilancia, la prevención y el control de criaderos de Aedes?

Se integran las fuentes a nivel de municipio, año y semana y se revisa la calidad de la tabla cruzada. La sección **21. Calidad de datos** presenta los tipos antes y después, hallazgos en `precipitacion_mm` y `casos_ins`, decisiones justificadas y conteos de los tratamientos.

**Ejecución:** en Google Colab, subir los dos CSV originales al panel de archivos y ejecutar todas las celdas en orden. En Jupyter local se puede iniciar desde la raíz del proyecto o desde `notebooks/`. Se requieren Python 3 y pandas. Los CSV originales se conservan; el resultado se guarda en un archivo aparte.


## 1. Importar pandas

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

print("Versión de pandas:", pd.__version__)


Versión de pandas: 2.3.3


## 2. Cargar los archivos CSV

Los identificadores se cargan como texto para conservar sus ceros iniciales. Por ejemplo, `05001` debe mantener sus cinco posiciones. Se conserva una copia de la lluvia original para auditar el efecto de la limpieza.


In [2]:
archivo_dengue = "ins_dengue_municipio_semana_2021_2025.csv"
archivo_lluvias = "lluvias_municipio_2021_2025.csv"
carpetas = [Path.cwd(), Path.cwd().parent, Path("/content")]
carpeta_datos = next(
    (carpeta for carpeta in carpetas
     if (carpeta / archivo_dengue).is_file() and (carpeta / archivo_lluvias).is_file()),
    None
)
if carpeta_datos is None:
    raise FileNotFoundError("Ubica los dos CSV originales en la carpeta del proyecto o súbelos a Colab.")

dengue = pd.read_csv(carpeta_datos / archivo_dengue, dtype={"codigo_municipio": "string"})
lluvias = pd.read_csv(
    carpeta_datos / archivo_lluvias,
    dtype={"codigoestacion": "string", "codigosensor": "string"}
)
lluvias_originales = lluvias.copy()
columnas_lluvias_originales = lluvias.columns.tolist()
print("Filas originales de dengue:", len(dengue))
print("Filas originales de lluvia:", len(lluvias))


Filas originales de dengue: 84895
Filas originales de lluvia: 1000


## 3. Revisar dengue

Se observan el número de filas y columnas, los primeros registros, los tipos de datos y los nombres de las columnas.

In [3]:
print("Dimensiones de dengue:", dengue.shape)
display(dengue.head())
print("Tipos de datos:")
display(dengue.dtypes)
print("Columnas:")
display(dengue.columns)


Dimensiones de dengue: (84895, 6)


,anio,semana,codigo_municipio,departamento,municipio,casos_ins
0,2021,1,05001,ANTIOQUIA,MEDELLIN,4
1,2021,1,05045,ANTIOQUIA,APARTADO,1
2,2021,1,05172,ANTIOQUIA,CHIGORODO,1
3,2021,1,05190,ANTIOQUIA,CISNEROS,2
4,2021,1,05250,ANTIOQUIA,EL BAGRE,1


Tipos de datos:


anio                         int64
semana                       int64
codigo_municipio    string[python]
departamento                object
municipio                   object
casos_ins                    int64
dtype: object

Columnas:


Index(['anio', 'semana', 'codigo_municipio', 'departamento', 'municipio',
       'casos_ins'],
      dtype='object')

## 4. Revisar lluvias

Esta fuente contiene observaciones por estación y fecha; todavía no tiene una fila por municipio y semana.

In [4]:
print("Dimensiones de lluvias:", lluvias.shape)
display(lluvias.head())
print("Tipos de datos:")
display(lluvias.dtypes)
print("Columnas:")
display(lluvias.columns)


Dimensiones de lluvias: (1000, 12)


,codigoestacion,codigosensor,fechaobservacion,valorobservado,nombreestacion,departamento,municipio,zonahidrografica,latitud,longitud,descripcionsensor,unidadmedida
0,0023085270,0240,2021-01-11T05:50:00.000,1.001,APTO J.M CORDOVA TX GPRS,ANTIOQUIA,RIONEGRO,MEDIO MAGDALENA,6.169000,-75.426000,Precipitacion,mm
1,0023085270,0240,2021-05-17T23:00:00.000,1.001,APTO J.M CORDOVA TX GPRS,ANTIOQUIA,RIONEGRO,MEDIO MAGDALENA,6.169000,-75.426000,Precipitacion,mm
2,3502500135,0240,2021-09-15T12:40:00.000,1.001,GUAYABETAL POLLO OLIMPICO - AUT,CUNDINAMARCA,GUAYABETAL,META,4.229669,-73.812625,Precipitacion,mm
3,0027015330,0240,2021-09-19T03:00:00.000,1.001,APTO OLAYA HERRERA - TX GPRS,ANTIOQUIA,MEDELLÍN,NECHÍ,6.221000,-75.589000,Precipitacion,mm
4,0035080050,0240,2021-12-24T19:40:00.000,1.001,PAEZ,BOYACÁ,PÁEZ,META,5.096000,-73.053000,Precipitacion,mm


Tipos de datos:


codigoestacion       string[python]
codigosensor         string[python]
fechaobservacion             object
valorobservado              float64
nombreestacion               object
departamento                 object
municipio                    object
zonahidrografica             object
latitud                     float64
longitud                    float64
descripcionsensor            object
unidadmedida                 object
dtype: object

Columnas:


Index(['codigoestacion', 'codigosensor', 'fechaobservacion', 'valorobservado',
       'nombreestacion', 'departamento', 'municipio', 'zonahidrografica',
       'latitud', 'longitud', 'descripcionsensor', 'unidadmedida'],
      dtype='object')

## 5. Preparar las columnas de texto y sus equivalencias

Se conservan los nombres originales y se crean claves sin espacios sobrantes, en mayúsculas y sin tildes en las vocales. Además, se resuelven variantes presentes en los CSV: `VALLE DEL CAUCA` / `VALLE`, `LA GUAJIRA` / `GUAJIRA`, `NORTE DE SANTANDER` / `NORTE SANTANDER` y `NARINO` / `NARIÑO`. La equivalencia `CARTAGENA DE INDIAS` / `CARTAGENA` se aplica únicamente en Bolívar. Los nombres de Bogotá se mantienen separados de Cundinamarca.


In [5]:
tabla_tildes = str.maketrans("ÁÉÍÓÚÜ", "AEIOUU")
equivalencias_departamento = {
    "VALLE DEL CAUCA": "VALLE",
    "LA GUAJIRA": "GUAJIRA",
    "NORTE DE SANTANDER": "NORTE SANTANDER",
    "NARINO": "NARIÑO"
}

def normalizar_texto(serie):
    return (
        serie.astype("string").str.strip().str.upper()
        .str.translate(tabla_tildes).str.replace(r"\s+", " ", regex=True)
        .replace("", pd.NA)
    )

resumen_equivalencias = []
for nombre_fuente, datos in [("Dengue", dengue), ("Lluvias", lluvias)]:
    departamento_base = normalizar_texto(datos["departamento"])
    municipio_base = normalizar_texto(datos["municipio"])
    datos["departamento_clave"] = departamento_base.replace(equivalencias_departamento)
    datos["municipio_clave"] = municipio_base
    cartagena = (
        datos["departamento_clave"].eq("BOLIVAR")
        & datos["municipio_clave"].eq("CARTAGENA DE INDIAS")
    )
    datos.loc[cartagena, "municipio_clave"] = "CARTAGENA"
    resumen_equivalencias.append({
        "Fuente": nombre_fuente,
        "Filas con equivalencia de departamento": int(
            departamento_base.ne(datos["departamento_clave"]).fillna(False).sum()
        ),
        "Filas con equivalencia de municipio": int(cartagena.sum())
    })

display(pd.DataFrame(resumen_equivalencias))
display(lluvias[["departamento", "departamento_clave", "municipio", "municipio_clave"]].drop_duplicates().head(12))


,Fuente,Filas con equivalencia de departamento,Filas con equivalencia de municipio
0,Dengue,0,0
1,Lluvias,18,4


,departamento,departamento_clave,municipio,municipio_clave
0,ANTIOQUIA,ANTIOQUIA,RIONEGRO,RIONEGRO
2,CUNDINAMARCA,CUNDINAMARCA,GUAYABETAL,GUAYABETAL
3,ANTIOQUIA,ANTIOQUIA,MEDELLÍN,MEDELLIN
4,BOYACÁ,BOYACA,PÁEZ,PAEZ
5,QUINDÍO,QUINDIO,ARMENIA,ARMENIA
10,SANTANDER,SANTANDER,BARRANCABERMEJA,BARRANCABERMEJA
13,NARIÑO,NARIÑO,ALDANA,ALDANA
16,RISARALDA,RISARALDA,PEREIRA,PEREIRA
23,BOYACÁ,BOYACA,PAJARITO,PAJARITO
26,BOGOTÁ,BOGOTA,BOGOTÁ D.C,BOGOTA D.C


## 6. Preparar fecha y valor de lluvias

Se cuentan las conversiones fallidas. Los valores que no puedan interpretarse quedan como faltantes y se reportan antes de la agregación.


In [6]:
fechas_originales = lluvias["fechaobservacion"].copy()
valores_originales = lluvias["valorobservado"].copy()
lluvias["fechaobservacion"] = pd.to_datetime(fechas_originales, errors="coerce")
lluvias["valorobservado"] = pd.to_numeric(valores_originales, errors="coerce")

print("Fechas no vacías que no se pudieron convertir:",
      int((fechas_originales.notna() & lluvias["fechaobservacion"].isna()).sum()))
print("Valores no vacíos que no se pudieron convertir:",
      int((valores_originales.notna() & lluvias["valorobservado"].isna()).sum()))


Fechas no vacías que no se pudieron convertir: 0
Valores no vacíos que no se pudieron convertir: 0


## 7. Filtrar precipitación

Primero se revisan los sensores y las unidades presentes. Se conservan las descripciones que contienen «PRECIPITACION» después de quitar la tilde, incluida la variante «GPRS - PRECIPITACIÓN», y cuya unidad es mm.

In [7]:
print("Descripciones de sensor:")
display(lluvias["descripcionsensor"].unique())
print("Unidades de medida:")
display(lluvias["unidadmedida"].unique())

lluvias = lluvias[
    lluvias["descripcionsensor"].astype(str).str.upper()
    .str.translate(tabla_tildes).str.contains("PRECIPITACION", na=False)
    & lluvias["unidadmedida"].astype(str).str.strip().str.lower().eq("mm")
].copy()

print("Observaciones de precipitación en mm:", len(lluvias))


Descripciones de sensor:


array(['Precipitacion', 'PRECIPITACIÓN', 'GPRS - PRECIPITACIÓN'],
      dtype=object)

Unidades de medida:


array(['mm'], dtype=object)

Observaciones de precipitación en mm: 1000


## 8. Crear año y semana para lluvias

La lluvia no contiene semana epidemiológica. Se obtiene la semana y el año ISO como **aproximación temporal**. Esta decisión puede desplazar observaciones respecto al calendario epidemiológico oficial, incluso fuera del intervalo de años en los límites de enero. Se reportan esos registros. La alineación con el calendario oficial queda pendiente antes de un análisis temporal de dengue y lluvias.


In [8]:
calendario = lluvias["fechaobservacion"].dt.isocalendar()
lluvias["anio"] = calendario.year.astype("Int64")
lluvias["semana"] = calendario.week.astype("Int64")
dengue["anio"] = pd.to_numeric(dengue["anio"], errors="coerce").astype("Int64")
dengue["semana"] = pd.to_numeric(dengue["semana"], errors="coerce").astype("Int64")

filas_antes_filtro = len(lluvias)
lluvias = lluvias.dropna(subset=[
    "codigoestacion", "departamento_clave", "municipio_clave",
    "anio", "semana", "valorobservado"
]).copy()
print("Observaciones sin datos necesarios para agregar:", filas_antes_filtro - len(lluvias))
print("Observaciones cuyo año ISO queda fuera de 2021-2025:",
      int((~lluvias["anio"].between(2021, 2025)).sum()))


Observaciones sin datos necesarios para agregar: 0
Observaciones cuyo año ISO queda fuera de 2021-2025: 5


## 9. Identificar y retirar duplicados exactos

Varias observaciones distintas en una semana pueden ser válidas. En cambio, una repetición de **todas las columnas originales** no aporta una nueva medición: se conserva la primera y se retiran las repeticiones para evitar sumar lluvia dos veces. También se comprueba que no queden varias filas con la misma estación, sensor y fecha; si persisten diferencias en sus demás campos, se requiere revisarlas antes de agregar.


In [9]:
lluvias_antes_deduplicar = lluvias.copy()
mascara_duplicados_lluvia = lluvias.duplicated(subset=columnas_lluvias_originales, keep="first")
numero_duplicados_lluvia = int(mascara_duplicados_lluvia.sum())
print("Repeticiones exactas retiradas:", numero_duplicados_lluvia)
display(lluvias.loc[mascara_duplicados_lluvia, columnas_lluvias_originales].head())
lluvias = lluvias.loc[~mascara_duplicados_lluvia].copy()

llave_observacion = ["codigoestacion", "codigosensor", "fechaobservacion"]
conflictos_observacion = lluvias.duplicated(subset=llave_observacion, keep=False)
print("Filas que aún comparten estación, sensor y fecha:", int(conflictos_observacion.sum()))
assert not conflictos_observacion.any(), "Revisar mediciones coincidentes antes de acumularlas."

columnas_estacion = [
    "codigoestacion", "departamento_clave", "municipio_clave", "anio", "semana"
]
conteo_observaciones = (
    lluvias.groupby(columnas_estacion, as_index=False).size()
    .rename(columns={"size": "numero_observaciones"})
)
print("Observaciones disponibles por estación y semana, después de retirar duplicados:")
display(conteo_observaciones["numero_observaciones"].describe())


Repeticiones exactas retiradas: 57


,codigoestacion,codigosensor,fechaobservacion,valorobservado,nombreestacion,departamento,municipio,zonahidrografica,latitud,longitud,descripcionsensor,unidadmedida
25,0026125061,0240,2025-02-04 08:40:00,1.002,AEROPUERTO EL EDEN -,QUINDÍO,ARMENIA,CAUCA,4.454722,-75.766389,PRECIPITACIÓN,mm
43,0035195050,0240,2024-10-30 20:20:00,1.003,CORINTO,BOYACÁ,PAJARITO,META,5.407278,-72.717222,PRECIPITACIÓN,mm
61,0026125061,0257,2024-10-04 16:22:00,1.005,AEROPUERTO EL EDEN -,QUINDÍO,ARMENIA,CAUCA,4.454722,-75.766389,GPRS - PRECIPITACIÓN,mm
63,0026125061,0257,2025-01-09 21:52:00,1.005,AEROPUERTO EL EDEN -,QUINDÍO,ARMENIA,CAUCA,4.454722,-75.766389,GPRS - PRECIPITACIÓN,mm
71,0026125061,0240,2024-10-22 01:20:00,1.006,AEROPUERTO EL EDEN -,QUINDÍO,ARMENIA,CAUCA,4.454722,-75.766389,PRECIPITACIÓN,mm


Filas que aún comparten estación, sensor y fecha: 0
Observaciones disponibles por estación y semana, después de retirar duplicados:


count    457.000000
mean       2.063457
std        2.249347
min        1.000000
25%        1.000000
50%        1.000000
75%        2.000000
max       19.000000
Name: numero_observaciones, dtype: float64

## 10. Agregar lluvia por estación y semana

Se suman las observaciones disponibles después de retirar los duplicados. **Esta suma no acredita una semana completa de precipitación:** la fuente tiene observaciones dispersas y no se conoce aquí su cobertura temporal esperada. Se conserva esta limitación al interpretar `precipitacion_mm`.


In [10]:
lluvia_estacion_semana = (
    lluvias.groupby(columnas_estacion, as_index=False)["valorobservado"].sum()
)
lluvia_estacion_semana = lluvia_estacion_semana.rename(
    columns={"valorobservado": "precipitacion_estacion_semana_mm"}
)
display(lluvia_estacion_semana.head())


,codigoestacion,departamento_clave,municipio_clave,anio,semana,precipitacion_estacion_semana_mm
0,0011045010,CHOCO,QUIBDO,2025,41,1.01
1,0011045010,CHOCO,QUIBDO,2025,42,1.01
2,0011045010,CHOCO,QUIBDO,2025,50,1.01
3,0011045010,CHOCO,QUIBDO,2025,51,1.01
4,0014015080,BOLIVAR,CARTAGENA,2021,22,1.01


## 11. Agregar lluvia por municipio y semana

Cuando existen varias estaciones dentro de un municipio, se utiliza el promedio de la precipitación semanal registrada por las estaciones para obtener un único valor municipal y evitar que los municipios con más estaciones acumulen lluvia de manera artificial.

In [11]:
llave_cruce = ["departamento_clave", "municipio_clave", "anio", "semana"]
lluvias_semana = (
    lluvia_estacion_semana.groupby(llave_cruce, as_index=False).agg(
        precipitacion_mm=("precipitacion_estacion_semana_mm", "mean"),
        numero_estaciones=("codigoestacion", "nunique")
    )
)


## 12. Revisar las lluvias agregadas

La llave municipal y semanal debe aparecer una sola vez en `lluvias_semana`.

In [12]:
print("Dimensiones de lluvias agregadas:", lluvias_semana.shape)
display(lluvias_semana.head())
print("Filas duplicadas por la llave del cruce:", lluvias_semana.duplicated(subset=llave_cruce).sum())


Dimensiones de lluvias agregadas: (454, 6)


,departamento_clave,municipio_clave,anio,semana,precipitacion_mm,numero_estaciones
0,AMAZONAS,LETICIA,2021,5,1.01,1
1,AMAZONAS,LETICIA,2022,9,1.01,1
2,AMAZONAS,LETICIA,2022,13,3.05,1
3,AMAZONAS,LETICIA,2022,15,1.01,1
4,AMAZONAS,LETICIA,2022,18,1.01,1


Filas duplicadas por la llave del cruce: 0


## 13. Verificar dengue

La llave debe identificar una sola fila. Si aparecen duplicados se detiene el proceso para revisar su origen y evitar sumar posibles repeticiones de notificaciones sin evidencia.


In [13]:
duplicados_dengue = int(dengue.duplicated(subset=llave_cruce).sum())
print("Duplicados de dengue por la llave del cruce:", duplicados_dengue)
assert dengue[llave_cruce].notna().all().all(), "Revisar valores faltantes en la llave de dengue."
assert duplicados_dengue == 0, "Revisar duplicados de dengue antes del cruce."
dengue_semana = dengue.copy()


Duplicados de dengue por la llave del cruce: 0


## 14. Llave del cruce

La llave es `departamento_clave`, `municipio_clave`, `anio` y `semana`. Ambas tablas tienen una fila por esta combinación. La equivalencia temporal sigue siendo aproximada porque la lluvia usa calendario ISO y dengue usa semana epidemiológica.


## 15. Filas antes del cruce

In [14]:
print("Filas de dengue:", len(dengue_semana))
print("Filas de lluvias:", len(lluvias_semana))


Filas de dengue: 84895
Filas de lluvias: 454


## 16. Cruzar las fuentes

El cruce izquierdo conserva los registros de dengue, aun cuando no encuentren información de lluvia.

In [15]:
datos_cruzados = pd.merge(
    dengue_semana,
    lluvias_semana,
    on=llave_cruce,
    how="left",
    validate="one_to_one",
    indicator=True
)
print("Estado del cruce:")
display(datos_cruzados["_merge"].value_counts())
datos_cruzados = datos_cruzados.drop(columns="_merge")


Estado del cruce:


_merge
left_only     84603
both            292
right_only        0
Name: count, dtype: int64

## 17. Verificar las filas después del cruce

In [16]:
print("Filas antes del cruce:", len(dengue_semana))
print("Filas después del cruce:", len(datos_cruzados))
print("Duplicados de la llave en el resultado:", int(datos_cruzados.duplicated(llave_cruce).sum()))
assert len(dengue_semana) == len(datos_cruzados), "El cruce cambió el número de filas."
assert not datos_cruzados.duplicated(llave_cruce).any(), "La llave del resultado debe ser única."
print("Dimensiones del resultado:", datos_cruzados.shape)
display(datos_cruzados.head())
print("Nulos por columna después del cruce:")
display(datos_cruzados.isna().sum().rename("nulos"))


Filas antes del cruce: 84895
Filas después del cruce: 84895
Duplicados de la llave en el resultado: 0
Dimensiones del resultado: (84895, 10)


,anio,semana,codigo_municipio,departamento,municipio,casos_ins,departamento_clave,municipio_clave,precipitacion_mm,numero_estaciones
0,2021,1,05001,ANTIOQUIA,MEDELLIN,4,ANTIOQUIA,MEDELLIN,NaN,NaN
1,2021,1,05045,ANTIOQUIA,APARTADO,1,ANTIOQUIA,APARTADO,NaN,NaN
2,2021,1,05172,ANTIOQUIA,CHIGORODO,1,ANTIOQUIA,CHIGORODO,NaN,NaN
3,2021,1,05190,ANTIOQUIA,CISNEROS,2,ANTIOQUIA,CISNEROS,NaN,NaN
4,2021,1,05250,ANTIOQUIA,EL BAGRE,1,ANTIOQUIA,EL BAGRE,NaN,NaN


Nulos por columna después del cruce:


anio                      0
semana                    0
codigo_municipio          0
departamento              0
municipio                 0
casos_ins                 0
departamento_clave        0
municipio_clave           0
precipitacion_mm      84603
numero_estaciones     84603
Name: nulos, dtype: int64

## 18. Registros sin lluvia

Estos registros corresponden a combinaciones de municipio, año y semana presentes en el dataset de dengue que no encontraron una observación equivalente en la fuente de precipitaciones.

In [17]:
sin_lluvia = datos_cruzados[
    datos_cruzados["precipitacion_mm"].isna()
]
print("Registros sin información de lluvia:", len(sin_lluvia))
display(sin_lluvia.head())


Registros sin información de lluvia: 84603


,anio,semana,codigo_municipio,departamento,municipio,casos_ins,departamento_clave,municipio_clave,precipitacion_mm,numero_estaciones
0,2021,1,05001,ANTIOQUIA,MEDELLIN,4,ANTIOQUIA,MEDELLIN,NaN,NaN
1,2021,1,05045,ANTIOQUIA,APARTADO,1,ANTIOQUIA,APARTADO,NaN,NaN
2,2021,1,05172,ANTIOQUIA,CHIGORODO,1,ANTIOQUIA,CHIGORODO,NaN,NaN
3,2021,1,05190,ANTIOQUIA,CISNEROS,2,ANTIOQUIA,CISNEROS,NaN,NaN
4,2021,1,05250,ANTIOQUIA,EL BAGRE,1,ANTIOQUIA,EL BAGRE,NaN,NaN


## 19. Porcentaje de cobertura

Se calcula qué proporción de registros de dengue encontró información de lluvia.

In [18]:
total = len(datos_cruzados)
con_lluvia = int(datos_cruzados["precipitacion_mm"].notna().sum())
porcentaje = con_lluvia / total * 100 if total else 0
print(f"Cobertura de lluvia: {con_lluvia:,} de {total:,} registros ({porcentaje:.2f} %).")
cobertura_anual = datos_cruzados.groupby("anio").agg(
    registros=("casos_ins", "size"),
    con_lluvia=("precipitacion_mm", "count")
)
cobertura_anual["cobertura_porcentaje"] = (
    100 * cobertura_anual["con_lluvia"] / cobertura_anual["registros"]
).round(2)
display(cobertura_anual)


Cobertura de lluvia: 292 de 84,895 registros (0.34 %).


,registros,con_lluvia,cobertura_porcentaje
anio,,,
2021,9781,65,0.66
2022,12406,66,0.53
2023,17586,66,0.38
2024,26061,65,0.25
2025,19061,30,0.16


**Interpretación y decisión:** un faltante de precipitación significa que no se encontró una observación coincidente; se conserva como faltante y se marca en la sección de calidad. La baja cobertura limita las comparaciones entre fuentes. Aun con las equivalencias de nombres corregidas, hace falta revisar la consulta que produjo el CSV de lluvia, su cobertura espacial y temporal y el calendario utilizado.


## 20. Verificar los años

In [19]:
print("Años presentes en el resultado:", sorted(datos_cruzados["anio"].dropna().unique()))


Años presentes en el resultado: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


## 21. Calidad de datos

Esta sección desarrolla la entrega de la clase asincrónica: se revisan los tipos **sobre la tabla ya cruzada**, se examinan al menos dos columnas relevantes para la pregunta del proyecto y se documentan los tratamientos y su justificación. El cruce conserva las filas y una llave única; sus faltantes se mantienen visibles como un problema de disponibilidad.

### 21.1. Tipos de columna antes y después

Los códigos son identificadores de texto; año, semana, casos y número de estaciones son enteros; la precipitación admite decimales. El entero anulable `Int64` permite representar las estaciones faltantes sin convertir el conteo en decimal. No hay una columna de fecha en la tabla cruzada: la fecha original de lluvia se convirtió antes de la agregación.


In [20]:
tipos_antes = datos_cruzados.dtypes.astype(str).rename("antes")
nulos_antes_tipos = datos_cruzados.isna().sum()
datos_cruzados = datos_cruzados.copy()

for columna in ["departamento", "municipio", "departamento_clave", "municipio_clave"]:
    datos_cruzados[columna] = datos_cruzados[columna].astype("string").str.strip()
datos_cruzados["codigo_municipio"] = (
    datos_cruzados["codigo_municipio"].astype("string").str.strip().str.zfill(5)
)
for columna in ["anio", "semana", "casos_ins", "numero_estaciones"]:
    serie = pd.to_numeric(datos_cruzados[columna], errors="coerce")
    # Una fracción no es un conteo entero: queda faltante para revisión.
    serie = serie.where(serie.isna() | serie.mod(1).eq(0))
    datos_cruzados[columna] = serie.astype("Int64")
datos_cruzados["precipitacion_mm"] = pd.to_numeric(
    datos_cruzados["precipitacion_mm"], errors="coerce"
).astype("float64")

auditoria_tipos = pd.concat([
    tipos_antes,
    datos_cruzados.dtypes.astype(str).rename("despues"),
    nulos_antes_tipos.rename("nulos_antes"),
    datos_cruzados.isna().sum().rename("nulos_despues")
], axis=1)
auditoria_tipos["nuevos_nulos_por_conversion"] = (
    auditoria_tipos["nulos_despues"] - auditoria_tipos["nulos_antes"]
)
display(auditoria_tipos)
print("Ejemplo de identificadores con cero inicial:")
display(datos_cruzados.loc[
    datos_cruzados["codigo_municipio"].str.startswith("0", na=False),
    ["codigo_municipio", "departamento", "municipio"]
].drop_duplicates().head())


,antes,despues,nulos_antes,nulos_despues,nuevos_nulos_por_conversion
anio,Int64,Int64,0,0,0
semana,Int64,Int64,0,0,0
codigo_municipio,string,string,0,0,0
departamento,object,string,0,0,0
municipio,object,string,0,0,0
casos_ins,int64,Int64,0,0,0
departamento_clave,string,string,0,0,0
municipio_clave,string,string,0,0,0
precipitacion_mm,float64,float64,84603,84603,0
numero_estaciones,float64,Int64,84603,84603,0


Ejemplo de identificadores con cero inicial:


,codigo_municipio,departamento,municipio
0,05001,ANTIOQUIA,MEDELLIN
1,05045,ANTIOQUIA,APARTADO
2,05172,ANTIOQUIA,CHIGORODO
3,05190,ANTIOQUIA,CISNEROS
4,05250,ANTIOQUIA,EL BAGRE


### 21.2. Diagnóstico de las columnas relevantes

`casos_ins` responde a la concentración de notificaciones y `precipitacion_mm` sirve para el contraste con lluvias. Se revisan sus distribuciones, faltantes y valores imposibles. También se validan códigos, años, semanas y conteos de estaciones. La ausencia de una fila de dengue en un municipio y semana no se interpreta automáticamente como cero casos.


In [21]:
display(datos_cruzados[["casos_ins", "precipitacion_mm"]].describe())
controles_dominio = pd.Series({
    "codigos_municipio_invalidos": int((~datos_cruzados["codigo_municipio"].str.fullmatch(r"\d{5}", na=False)).sum()),
    "anios_invalidos_o_faltantes": int((~datos_cruzados["anio"].between(2021, 2025).fillna(False)).sum()),
    "semanas_invalidas_o_faltantes": int((~datos_cruzados["semana"].between(1, 53).fillna(False)).sum()),
    "casos_negativos": int(datos_cruzados["casos_ins"].lt(0).sum()),
    "casos_faltantes": int(datos_cruzados["casos_ins"].isna().sum()),
    "precipitaciones_negativas": int(datos_cruzados["precipitacion_mm"].lt(0).sum()),
    "estaciones_no_positivas": int(datos_cruzados["numero_estaciones"].le(0).sum())
}, name="filas_detectadas")
display(controles_dominio)


,casos_ins,precipitacion_mm
count,84895.0,292.000000
mean,7.894658,2.157425
std,29.226415,2.279972
min,1.0,1.001000
25%,1.0,1.010000
50%,2.0,1.020000
75%,6.0,2.040000
max,1295.0,19.300000


codigos_municipio_invalidos      0
anios_invalidos_o_faltantes      0
semanas_invalidas_o_faltantes    0
casos_negativos                  0
casos_faltantes                  0
precipitaciones_negativas        0
estaciones_no_positivas          0
Name: filas_detectadas, dtype: int64

### 21.3. Hallazgos y tratamiento de `precipitacion_mm`

**Hallazgo 1: precipitación inflada por registros duplicados.** En la fuente se encontraron 57 repeticiones exactas. **Decisión:** retirar esas repeticiones antes de agregar y recalcular la precipitación. **Justificación:** la misma medición repetida no debe aumentar la lluvia. A continuación se compara, sobre las filas cruzadas, el resultado que se habría obtenido conservando los duplicados con el resultado corregido. La bandera `precipitacion_corregida_duplicados` conserva la trazabilidad de las filas afectadas.


In [22]:
lluvia_estacion_sin_limpieza = (
    lluvias_antes_deduplicar.groupby(columnas_estacion, as_index=False)["valorobservado"].sum()
)
lluvia_municipio_sin_limpieza = (
    lluvia_estacion_sin_limpieza.groupby(llave_cruce, as_index=False)["valorobservado"].mean()
    .rename(columns={"valorobservado": "precipitacion_antes_mm"})
)
comparacion_precipitacion = datos_cruzados[
    llave_cruce + ["municipio", "precipitacion_mm"]
].merge(lluvia_municipio_sin_limpieza, on=llave_cruce, how="left", validate="one_to_one")
comparacion_precipitacion["corregida"] = (
    comparacion_precipitacion["precipitacion_antes_mm"].notna()
    & comparacion_precipitacion["precipitacion_mm"].notna()
    & (comparacion_precipitacion["precipitacion_antes_mm"]
       - comparacion_precipitacion["precipitacion_mm"]).abs().gt(1e-9)
)
datos_cruzados["precipitacion_corregida_duplicados"] = (
    comparacion_precipitacion["corregida"].astype("boolean").to_numpy()
)
datos_cruzados["precipitacion_corregida_duplicados"] = (
    datos_cruzados["precipitacion_corregida_duplicados"].astype("boolean")
)
filas_lluvia_corregidas = int(comparacion_precipitacion["corregida"].sum())
print("Filas del resultado cuya precipitación cambió:", filas_lluvia_corregidas)
display(comparacion_precipitacion.loc[
    comparacion_precipitacion["corregida"],
    ["departamento_clave", "municipio", "anio", "semana", "precipitacion_antes_mm", "precipitacion_mm"]
].head(10))


Filas del resultado cuya precipitación cambió: 14


,departamento_clave,municipio,anio,semana,precipitacion_antes_mm,precipitacion_mm
60179,QUINDIO,ARMENIA,2024,40,2.010,1.005
60194,RISARALDA,LA VIRGINIA,2024,40,4.064,2.032
60641,QUINDIO,ARMENIA,2024,41,2.016,1.008
61272,CASANARE,YOPAL,2024,42,2.020,1.010
61599,QUINDIO,ARMENIA,2024,43,2.012,1.006
63464,RISARALDA,LA VIRGINIA,2024,47,14.224,7.112
64038,CASANARE,YOPAL,2024,48,2.020,1.010
64353,RISARALDA,LA VIRGINIA,2024,49,8.128,4.064
64806,RISARALDA,LA VIRGINIA,2024,50,12.192,6.096
65696,RISARALDA,LA VIRGINIA,2024,52,12.192,6.096


**Hallazgo 2: datos de lluvia ausentes y cobertura insuficiente.** La mayoría de filas cruzadas no tiene precipitación. Además, el CSV original contiene exactamente 1.000 observaciones, ordenadas de menor a mayor por `valorobservado`, con un rango de 1,001 a 1,020 mm. **Decisión:** conservar los faltantes, crear `sin_dato_lluvia` y marcar todas las observaciones disponibles como `lluvia_pendiente_validar_fuente`. **Justificación:** un faltante no demuestra ausencia de lluvia y esta extracción no acredita una serie representativa ni una semana completa. Debe revisarse si la consulta original tuvo un límite, filtro u ordenamiento; esto es una hipótesis pendiente, no un error de magnitud confirmado. El rango estrecho por sí solo tampoco justifica cambiar las mediciones.


In [23]:
valores_lluvia_original = pd.to_numeric(lluvias_originales["valorobservado"], errors="coerce")
diagnostico_fuente_lluvia = pd.Series({
    "filas_originales": len(lluvias_originales),
    "valores_numericos_distintos": valores_lluvia_original.nunique(),
    "minimo_mm": valores_lluvia_original.min(),
    "maximo_mm": valores_lluvia_original.max(),
    "orden_ascendente_por_valor": valores_lluvia_original.is_monotonic_increasing
}, name="diagnostico")
display(diagnostico_fuente_lluvia)
datos_cruzados["sin_dato_lluvia"] = datos_cruzados["precipitacion_mm"].isna().astype("boolean")
datos_cruzados["lluvia_pendiente_validar_fuente"] = (
    datos_cruzados["precipitacion_mm"].notna().astype("boolean")
)
print("Filas sin dato de lluvia:", int(datos_cruzados["sin_dato_lluvia"].sum()))
print("Filas con lluvia pendiente de validar en la fuente:",
      int(datos_cruzados["lluvia_pendiente_validar_fuente"].sum()))


filas_originales                1000
valores_numericos_distintos       20
minimo_mm                      1.001
maximo_mm                       1.02
orden_ascendente_por_valor      True
Name: diagnostico, dtype: object

Filas sin dato de lluvia: 84603
Filas con lluvia pendiente de validar en la fuente: 292


### 21.4. Hallazgo y tratamiento de `casos_ins`

**Hallazgo 3: valores altos de notificaciones.** El máximo es 1.295 casos en Cali, semana 21 de 2024. Para detectar extremos se usa el rango intercuartílico (IQR) **dentro de cada municipio** en el periodo observado: se marcan valores superiores a Q3 + 1,5 × IQR cuando existen al menos ocho registros y un IQR mayor que cero. Es una señal exploratoria, sensible a tendencias y cambios entre años.

**Decisión:** conservar los valores y crear `casos_atipicos_revisar`, mostrando además las semanas cercanas al máximo. **Justificación:** un pico puede corresponder a un aumento real de notificaciones y es relevante para la pregunta del proyecto. Las semanas vecinas de Cali también tienen cifras altas; no hay evidencia suficiente para borrar o reemplazar el máximo. La bandera pide contrastar con la fuente original y el contexto temporal; no clasifica el registro como error confirmado.


In [24]:
grupos_casos = datos_cruzados.groupby("codigo_municipio")["casos_ins"]
q1_municipio = grupos_casos.transform(lambda serie: serie.quantile(0.25))
q3_municipio = grupos_casos.transform(lambda serie: serie.quantile(0.75))
iqr_municipio = q3_municipio - q1_municipio
suficientes_registros = grupos_casos.transform("count").ge(8)
limite_superior = q3_municipio + 1.5 * iqr_municipio
datos_cruzados["casos_atipicos_revisar"] = (
    suficientes_registros & iqr_municipio.gt(0)
    & datos_cruzados["casos_ins"].gt(limite_superior)
).fillna(False).astype("boolean")

print("Filas con casos altos marcadas para revisión:",
      int(datos_cruzados["casos_atipicos_revisar"].sum()))
display(datos_cruzados.loc[
    datos_cruzados["casos_atipicos_revisar"],
    ["codigo_municipio", "municipio", "anio", "semana", "casos_ins", "casos_atipicos_revisar"]
].sort_values("casos_ins", ascending=False).head(10))

registro_maximo = datos_cruzados.loc[datos_cruzados["casos_ins"].idxmax()]
contexto_maximo = datos_cruzados[
    datos_cruzados["codigo_municipio"].eq(registro_maximo["codigo_municipio"])
    & datos_cruzados["anio"].eq(registro_maximo["anio"])
    & datos_cruzados["semana"].between(registro_maximo["semana"] - 2, registro_maximo["semana"] + 2)
]
print("Semanas cercanas al máximo:")
display(contexto_maximo[["municipio", "anio", "semana", "casos_ins", "casos_atipicos_revisar"]].sort_values("semana"))


Filas con casos altos marcadas para revisión: 5632


,codigo_municipio,municipio,anio,semana,casos_ins,casos_atipicos_revisar
50043,76001,CALI,2024,21,1295,True
51153,76001,CALI,2024,23,1239,True
50601,76001,CALI,2024,22,1234,True
51730,76001,CALI,2024,24,1230,True
52307,76001,CALI,2024,25,1219,True
48402,76001,CALI,2024,18,1134,True
46328,76001,CALI,2024,14,1103,True
47379,76001,CALI,2024,16,1086,True
49487,76001,CALI,2024,20,1083,True
48933,76001,CALI,2024,19,1056,True


Semanas cercanas al máximo:


,municipio,anio,semana,casos_ins,casos_atipicos_revisar
48933,CALI,2024,19,1056,True
49487,CALI,2024,20,1083,True
50043,CALI,2024,21,1295,True
50601,CALI,2024,22,1234,True
51153,CALI,2024,23,1239,True


### 21.5. Resumen de tratamientos y porcentajes

Se desarrolla el reto opcional de contar los tratamientos y su porcentaje. Se distingue el total de observaciones originales de lluvia del total de filas cruzadas. Las banderas pueden coincidir en una misma fila, por lo que sus conteos no deben sumarse como si fueran registros diferentes. Los valores altos de dengue se conservaron y los faltantes de precipitación se mantuvieron.


In [25]:
resumen_calidad = pd.DataFrame([
    ["Lluvia original", "Repeticiones exactas retiradas", numero_duplicados_lluvia, len(lluvias_originales), "corregida"],
    ["precipitacion_mm", "Filas recalculadas por duplicados", filas_lluvia_corregidas, len(datos_cruzados), "corregida"],
    ["precipitacion_mm", "Faltantes conservados y marcados", int(datos_cruzados["sin_dato_lluvia"].sum()), len(datos_cruzados), "pendiente de revisar"],
    ["precipitacion_mm", "Datos disponibles con fuente pendiente de validar", int(datos_cruzados["lluvia_pendiente_validar_fuente"].sum()), len(datos_cruzados), "pendiente de revisar"],
    ["casos_ins", "Valores altos conservados y marcados", int(datos_cruzados["casos_atipicos_revisar"].sum()), len(datos_cruzados), "pendiente de revisar"]
], columns=["Columna o fuente", "Tratamiento", "Cantidad", "Total de referencia", "Estado"])
resumen_calidad["Porcentaje"] = (
    100 * resumen_calidad["Cantidad"] / resumen_calidad["Total de referencia"]
).round(2)
display(resumen_calidad)


,Columna o fuente,Tratamiento,Cantidad,Total de referencia,Estado,Porcentaje
0,Lluvia original,Repeticiones exactas retiradas,57,1000,corregida,5.70
1,precipitacion_mm,Filas recalculadas por duplicados,14,84895,corregida,0.02
2,precipitacion_mm,Faltantes conservados y marcados,84603,84895,pendiente de revisar,99.66
3,precipitacion_mm,Datos disponibles con fuente pendiente de validar,292,84895,pendiente de revisar,0.34
4,casos_ins,Valores altos conservados y marcados,5632,84895,pendiente de revisar,6.63


## 22. Diccionario de datos actualizado

Incluye las columnas originales de la tabla integrada y las cuatro banderas de calidad. El tipo corresponde a la tabla en memoria; CSV no conserva los tipos de pandas. El estado de precipitación y casos mantiene visibles las revisiones pendientes.


In [26]:
definiciones = [
    ["anio", "Ambas fuentes", "Año epidemiológico de dengue; año ISO de lluvia usado como aproximación.", "pendiente de revisar"],
    ["semana", "Ambas fuentes", "Semana epidemiológica de dengue y semana ISO de lluvia; falta alinear calendarios.", "pendiente de revisar"],
    ["codigo_municipio", "INS", "Identificador municipal de cinco posiciones, conservado como texto con ceros iniciales.", "sin problemas detectados"],
    ["departamento", "INS", "Nombre original del departamento de dengue.", "sin problemas detectados"],
    ["municipio", "INS", "Nombre original del municipio de dengue.", "sin problemas detectados"],
    ["casos_ins", "INS", "Número de notificaciones; los extremos se conservan y se marcan para revisión.", "pendiente de revisar"],
    ["departamento_clave", "Ambas fuentes", "Departamento normalizado y con equivalencias de nombres aplicadas.", "corregida"],
    ["municipio_clave", "Ambas fuentes", "Municipio normalizado; Cartagena de Indias se unifica con Cartagena en Bolívar.", "corregida"],
    ["precipitacion_mm", "CSV de lluvia", "Promedio entre estaciones de la suma de observaciones disponibles por semana, en mm; no acredita un total semanal completo.", "pendiente de revisar"],
    ["numero_estaciones", "CSV de lluvia", "Estaciones con observaciones utilizadas en esa combinación municipal y semanal; faltante si no hubo cruce.", "sin problemas detectados"],
    ["precipitacion_corregida_duplicados", "Control de calidad", "Verdadero si retirar duplicados exactos modificó la precipitación de la fila.", "corregida"],
    ["sin_dato_lluvia", "Control de calidad", "Verdadero cuando no hay dato de precipitación en la tabla cruzada.", "sin problemas detectados"],
    ["lluvia_pendiente_validar_fuente", "Control de calidad", "Verdadero para lluvia disponible cuya extracción y representatividad deben revisarse.", "pendiente de revisar"],
    ["casos_atipicos_revisar", "Control de calidad", "Verdadero si casos supera Q3 + 1,5 IQR municipal con al menos ocho datos e IQR positivo; señal exploratoria.", "pendiente de revisar"]
]
diccionario_datos = pd.DataFrame(definiciones, columns=["Columna", "Fuente", "Significado", "Estado"])
diccionario_datos.insert(1, "Tipo de dato", diccionario_datos["Columna"].map(datos_cruzados.dtypes.astype(str)))
display(diccionario_datos)


,Columna,Tipo de dato,Fuente,Significado,Estado
0,anio,Int64,Ambas fuentes,Año epidemiológico de dengue; año ISO de lluvi...,pendiente de revisar
1,semana,Int64,Ambas fuentes,Semana epidemiológica de dengue y semana ISO d...,pendiente de revisar
2,codigo_municipio,string,INS,"Identificador municipal de cinco posiciones, c...",sin problemas detectados
3,departamento,string,INS,Nombre original del departamento de dengue.,sin problemas detectados
4,municipio,string,INS,Nombre original del municipio de dengue.,sin problemas detectados
5,casos_ins,Int64,INS,Número de notificaciones; los extremos se cons...,pendiente de revisar
6,departamento_clave,string,Ambas fuentes,Departamento normalizado y con equivalencias d...,corregida
7,municipio_clave,string,Ambas fuentes,Municipio normalizado; Cartagena de Indias se ...,corregida
8,precipitacion_mm,float64,CSV de lluvia,Promedio entre estaciones de la suma de observ...,pendiente de revisar
9,numero_estaciones,Int64,CSV de lluvia,Estaciones con observaciones utilizadas en esa...,sin problemas detectados


## 23. Verificación final

Se comprueban las condiciones del cruce y los dominios de los campos. Los nulos de lluvia y estaciones se permiten cuando están representados por la bandera de ausencia. Los valores altos marcados permanecen en el resultado.


In [27]:
assert len(datos_cruzados) == len(dengue_semana)
assert not datos_cruzados.duplicated(llave_cruce).any()
assert datos_cruzados[llave_cruce].notna().all().all()
assert datos_cruzados["codigo_municipio"].str.fullmatch(r"\d{5}", na=False).all()
assert datos_cruzados["anio"].between(2021, 2025).all()
assert datos_cruzados["semana"].between(1, 53).all()
assert datos_cruzados["casos_ins"].notna().all()
assert datos_cruzados["casos_ins"].ge(0).all()
assert datos_cruzados["precipitacion_mm"].dropna().ge(0).all()
assert datos_cruzados["numero_estaciones"].dropna().ge(1).all()
assert datos_cruzados["sin_dato_lluvia"].equals(datos_cruzados["precipitacion_mm"].isna().astype("boolean"))
assert datos_cruzados["numero_estaciones"].isna().equals(datos_cruzados["precipitacion_mm"].isna())
assert set(diccionario_datos["Columna"]) == set(datos_cruzados.columns)
assert datos_cruzados["casos_ins"].sum() == dengue_semana["casos_ins"].sum()
print("Verificaciones completadas: filas, llave, tipos, dominios, banderas y diccionario consistentes.")
print("Dimensiones finales:", datos_cruzados.shape)
print("Nulos finales por columna:")
display(datos_cruzados.isna().sum().rename("nulos"))


Verificaciones completadas: filas, llave, tipos, dominios, banderas y diccionario consistentes.
Dimensiones finales: (84895, 14)
Nulos finales por columna:


anio                                      0
semana                                    0
codigo_municipio                          0
departamento                              0
municipio                                 0
casos_ins                                 0
departamento_clave                        0
municipio_clave                           0
precipitacion_mm                      84603
numero_estaciones                     84603
precipitacion_corregida_duplicados        0
sin_dato_lluvia                           0
lluvia_pendiente_validar_fuente           0
casos_atipicos_revisar                    0
Name: nulos, dtype: int64

## 24. Guardar el resultado revisado

Se exporta después de las correcciones y las banderas de calidad, en la misma carpeta donde están los CSV de entrada. Para volver a leer el archivo, debe indicarse `dtype={"codigo_municipio": "string"}` y así conservar los ceros iniciales. La entrega de la clase es este notebook ejecutado y actualizado en el repositorio.


In [28]:
ruta_resultado = carpeta_datos / "dengue_lluvias_municipio_semana_2021_2025.csv"
datos_cruzados.to_csv(ruta_resultado, index=False)
print("Archivo generado:", ruta_resultado.name)
print("Filas exportadas:", len(datos_cruzados))
print("Columnas exportadas:", len(datos_cruzados.columns))


Archivo generado: dengue_lluvias_municipio_semana_2021_2025.csv
Filas exportadas: 84895
Columnas exportadas: 14


## Conclusión

Se conservaron las 84.895 filas de dengue y una llave única por departamento, municipio, año y semana. Se normalizaron nombres territoriales, se preservaron los identificadores como texto y se retiraron 57 repeticiones exactas de lluvia antes de agregar. La sección de calidad muestra el efecto de esa corrección, los tipos antes y después y los porcentajes de registros corregidos o marcados.

Se trataron hallazgos en dos columnas relevantes: en `precipitacion_mm` se recalcularon valores afectados por duplicados y se conservaron y marcaron los faltantes; en `casos_ins` se conservaron los valores altos y se marcaron para revisión con un criterio municipal. El máximo de Cali se contextualizó con semanas cercanas sin asumir que fuera un error.

La cobertura de lluvia continúa siendo muy baja. Quedan pendientes validar la extracción original de las 1.000 observaciones, su representatividad y cobertura temporal y la alineación del calendario ISO con el epidemiológico. Por eso, la suma de lluvia disponible no debe interpretarse como un total semanal completo y este cruce todavía no sustenta conclusiones sobre la relación entre lluvias y dengue ni sobre causalidad.
